In [1]:
import re
import json
import random
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from collections import Counter
api_key = 'api-key'
client = OpenAI(api_key = api_key)

In [2]:
# df_train.to_csv('../result/train_GPT4o_inference_en.csv',sep=',',index=False)
df_train = pd.read_csv('../result/train_GPT4o_inference_en.csv',sep=',')
df_join = df_train
# df_val = pd.read_csv('../result/val_GPT4o_inference_en.csv',sep=',')
# df_join = pd.concat([df_train,df_val])
# df_join = df_join.reset_index(drop=True)
print(len(df_join))
df_join.head()

108


,id,t,n,m,text,response
0,56344,T4,N3,M0,Complete atelectasis of the left upper lobe.\n...,"The tumor is 74 mm, classifying it as T4 due t..."
1,133166,T1c,N0,M0,Tumorous lesion with a diameter of 30 mm in th...,"The lesion measures 30 mm, classifying it as T..."
2,165742,T3,N0,M0,Pulmonary mass with a diameter of 5 cm in the ...,"The pulmonary mass measures 5 cm, which fits t..."
3,404886,T4,N2,M1c,Tumorous lesion with a maximum diameter of 72 ...,"The tumor measures 72mm, exceeding 70mm, class..."
4,463397,T2b,N0,M0,Prominent emphysematous changes. Irregular lob...,"The tumor measures 47mm, indicating T2b as it'..."


In [3]:
test_data = pd.read_csv('../test_data/test_include_TextAns.csv')
print(len(test_data))
test_data.head()

135


,id,text,status,t,n,m
0,147290,"37 mm mass in the left pulmonary hilum, suspic...",val,T2a,N0,M0
1,241752,Focal ground-glass opacity measuring 14 x 15 m...,val,Tis,N0,M0
2,467592,Infiltrative shadows in the right middle lobe ...,test,T4,N1,M0
3,612334,Mass-like lesion approximately 3 cm in size in...,test,T1c,N0,M0
4,876951,"Tumor with a maximum diameter of 37 mm, mainly...",val,T2a,N0,M0


In [4]:
inference_process = []

# 確保欄位名稱正確無誤
for _, row in df_join.iterrows():
    t = 'example:#{} \nInput:{} \nReasoning: {} \nOutput: {{"T":{},"N":{},"M":{}}}'.format(
        row['id'],
        row['text'],  # 確保 'text' 是正確的欄位名稱
        row['response'],
        row['t'], row['n'], row['m']
    )
    inference_process.append(t)
inference_df = pd.DataFrame()
inference_df['id'] = df_join['id']
inference_df['inference_process'] = inference_process

print(len(inference_df))
inference_df.head()

108


,id,inference_process
0,56344,example:#56344 \nInput:Complete atelectasis of...
1,133166,example:#133166 \nInput:Tumorous lesion with a...
2,165742,example:#165742 \nInput:Pulmonary mass with a ...
3,404886,example:#404886 \nInput:Tumorous lesion with a...
4,463397,example:#463397 \nInput:Prominent emphysematou...


In [5]:
import random
def parse_tnm_from_text(text):
    # TNM: {"T":"T2a","N":"N1","M":"M0"} 或 "TNM":{"T":"T2a","N":"N1","M":"M0"}
    pattern_with_tnm = r'TNM["\']?:\s*{\s*["\']?T["\']?\s*:\s*["\']?([^"\']+)["\']?\s*,\s*["\']?N["\']?\s*:\s*["\']?([^"\']+)["\']?\s*,\s*["\']?M["\']?\s*:\s*["\']?([^"\']+)["\']?\s*}'
    # 第二個正則：匹配無 "TNM" 字樣的字串，如：
    # {"T":"T2a","N":"N1","M":"M0"}
    pattern_without_tnm = r'{\s*["\']?T["\']?\s*:\s*["\']?([^"\']+)["\']?\s*,\s*["\']?N["\']?\s*:\s*["\']?([^"\']+)["\']?\s*,\s*["\']?M["\']?\s*:\s*["\']?([^"\']+)["\']?\s*}'
    match = re.search(pattern_with_tnm, text)
    if not match:
        match = re.search(pattern_without_tnm, text)
    if match:
        t = match.group(1)
        n = match.group(2)
        m = match.group(3)
        return {'TNM': {'T': t, 'N': n, 'M': m}}
    else:
        return None

def get_example(n, exclude_id):
    exclude_id = int(exclude_id)
    id_include = inference_df['id'].tolist()
    if exclude_id in id_include:
        exclude_id_df = inference_df[inference_df['id']!=int(exclude_id)]
        exclude_id_df.reset_index(drop=True, inplace=True)
    else:
        exclude_id_df = inference_df
    # print(len(exclude_id_df))
    inference_process = exclude_id_df['inference_process'].tolist()
    random_sample = random.sample(inference_process, min(n, len(inference_process)))
    return random_sample

random_sample = get_example(1,'147290')
for i, example in enumerate(random_sample, start=1):
    print(f"{example}\n")

example:#1538432 
Input:There is a tumor with a maximum diameter of 47 mm in the upper left lobe. It has infiltrated the lower lobe, crossing the interlobar pleura. The hilar lymph nodes are fused with the tumor. No enlargement of the mediastinal lymph nodes is observed. There is no pleural effusion. No liver or adrenal metastasis is observed. No significant abnormalities are observed in the visualized abdominal organs. 
Reasoning: The tumor is T2b because it measures 47 mm, falling between >4 cm (40 mm) and ?? cm (50 mm), and involves pleural infiltration. It is N1 as it involves ipsilateral (hilar) lymph nodes. M0 is appropriate as there is no distant metastasis or pleural effusion observed. 
Output: {"T":T2b,"N":N1,"M":M0}



Use train inference to randomly select examples as fewshots

In [6]:
def classify_tnm(case, n_shot, case_id):
    example_case = get_example(n=n_shot , exclude_id=case_id)
    additional_cases = "\n".join(example_case)
    # print(additional_cases)
    completion = client.chat.completions.create(
      model="gpt-4o",
      messages = [
    {
        "role": "system",
        "content": f"""You are a medical AI assistant specializing in tumor staging. Your task is to determine the TNM stage based on the provided clinical text descriptions.

        **Key Instructions**:
        - Tumor size should be specified in cm or mm.
        - If lymph node abnormalities are present but uncertain, classify them as **N0**.
        - Tumors without evidence of invasion should be prioritized for **Tis** classification, even if the size approaches criteria for other stages like **T1b**.
        - Do not classify tumors as **T1b** based solely on size; clear pathological evidence of invasion is required.
        - If there is uncertainty, always select the most conservative stage.

        **I. Primary Tumor (T) Staging**:
        - **T0**: No primary tumor.
        - **Tis** (Carcinoma in situ):
          - Tumor ≤3cm (30mm), limited to the epithelium, no evidence of invasion into the underlying tissue or stroma.
          - **Key point**: Tumors without invasive features must be classified as Tis.
        - **T1mi** (Minimally invasive adenocarcinoma):
          - Tumor ≤3cm (30mm) with minimal invasion (≤5mm into stroma).
          - **Key point**: Requires pathological confirmation of invasion; otherwise, classify as Tis.
        - **T1a**: Tumor ≤1cm (10mm), no deeper invasion.
        - **T1b**: Tumor >1cm (10mm) but ≤2cm (20mm), with evidence of invasion.
        - **T1c**: Tumor >2cm (20mm) but ≤3cm (30mm).
        - **T2a**: Tumor >3cm (30mm) but <4cm (40mm) or tumor involving visceral pleura, main bronchus (not carina), or atelectasis to hilum.
        - **T2b**: Tumor >4cm (40mm) but ≤5cm (50mm) or tumor involving visceral pleura, main bronchus (not carina), or atelectasis to hilum.
        - **T3**: Tumor >5cm (50mm) but ≤7cm (70mm) or invading chest wall, pericardium, phrenic nerve, or separate tumor nodule(s) in the same lobe.
        - **T4**: Tumor >7cm (70mm) or invading mediastinum, diaphragm, heart, great vessels, recurrent laryngeal nerve, trachea, esophagus, spine; or tumor nodule(s) in a different ipsilateral lobe.

        - In cases of **uncertainty** regarding invasion or measurements, choose the **most conservative stage**.
        - When describing tumor dimensions in the report, always distinguish between **total size** and **solid component size**, and use the solid component for staging where applicable.

        **II. Lymph Node Metastasis (N) Staging**:
        - **N0**: No regional lymph node metastasis.
          - Use only when lymph nodes are clearly normal without any suspicion of metastasis.
        - **N1**: Ipsilateral peribronchial or hilar lymph node metastasis.
          - Classify as **N1** if there is mention of "prominent hilar lymph nodes" or "suspicion of metastasis in the hilar nodes."
        - **N2**: Ipsilateral mediastinal or subcarinal lymph node metastasis.
          - Classify as **N2** if there is mention of "enlarged mediastinal lymph nodes" or "cannot rule out metastasis in mediastinal lymph nodes."
        - **N3**: Contralateral mediastinal, contralateral hilar, scalene, or supraclavicular lymph node metastasis.

        **III. Distant Metastasis (M) Staging**
        - **M0**: No distant metastasis.
          - **Key point**: Use this classification only when there is no evidence of distant metastasis, including pleural dissemination, malignant pleural effusion, or any organ involvement.
        - **M1a**: Pleural dissemination (malignant pleural or pericardial effusion, pleural nodules), or separate tumor nodules in the contralateral lobe.
          - **Key point**:
            - Include cases with pleural effusion confirmed as malignant or pleural-based nodules.
            - Also include separate nodules identified in the contralateral lung.
            - If additional evidence suggests metastases in other organs, classify as **M1c** instead.
        - **M1b**: Single-organ distant metastasis (outside lung/pleura).
          - **Key point**:
            - Classify as M1b if a single organ (e.g., liver, bone, brain, or adrenal gland) shows definitive evidence of metastasis.
            - Ensure that no metastases are present in other organs.
            - If multiple organs are involved, reclassify as **M1c**.
        - **M1c**: Multiple-organ distant metastases.
          - **Key point**:
            - Use this classification when metastases are identified in two or more organs (e.g., bone, liver, adrenal glands).
            - Include cases with pleural dissemination if other organs are also involved.
            - Examples include bilateral adrenal gland metastases or widespread bone lesions.

        If uncertain, select the stage that best fits the situation.\n\n

        **Output Requirement**:
        - Provide the TNM code in JSON format: {{'TNM': {{'T': '', 'N': '', 'M': ''}}}}.
        - Do not include reasoning, explanations, or any additional text outside the JSON format.

        **[Few-shot Examples]**\n
          {additional_cases}\n\n
        **[Testing Data]**\n
          The following are samples to be analyzed. Please provide the TNM staging results using the same logic."""
    },
    {
        "role": "user",
        "content": f"Input: {case}\n\nProvide the TNM staging result using the above logic."
    }
],
    temperature=0.2,
    max_tokens=50,
    top_p=0.8,
    frequency_penalty=0,
    presence_penalty=0,
    n=1
    )
    response = completion.choices[0].message.content.strip()
    result = parse_tnm_from_text(response)
    #json.loads(completion.choices[0].message.content.strip())

    return response, result

In [ ]:
def classify_tnm(case, n_shot, case_id):
    example_case = get_example(n=n_shot , exclude_id=case_id)
    additional_cases = "\n".join(example_case)
    # print(additional_cases)
    completion = client.chat.completions.create(
      model="gpt-4o",
      messages = [
    {
        "role": "system",
        "content": f"""You are a medical AI assistant specializing in tumor staging. Your task is to determine the TNM stage based on the provided clinical text descriptions.

        **Key Instructions**:
        - Tumor size should be specified in cm or mm.
        - If lymph node abnormalities are present but uncertain, classify them as **N0**.
        - Tumors without evidence of invasion should be prioritized for **Tis** classification, even if the size approaches criteria for other stages like **T1b**.
        - Do not classify tumors as **T1b** based solely on size; clear pathological evidence of invasion is required.
        - If there is uncertainty, always select the most conservative stage.

        **I. Primary Tumor (T) Staging**:
        - **T0**: No primary tumor.
        - **Tis** (Carcinoma in situ):
          - Tumor ≤3cm (30mm), limited to the epithelium, no evidence of invasion into the underlying tissue or stroma.
          - **Key point**: Tumors without invasive features must be classified as Tis.
        - **T1mi** (Minimally invasive adenocarcinoma):
          - Tumor ≤3cm (30mm) with minimal invasion (≤5mm into stroma).
          - **Key point**: Requires pathological confirmation of invasion; otherwise, classify as Tis.
        - **T1a**: Tumor ≤1cm (10mm), no deeper invasion.
        - **T1b**: Tumor >1cm (10mm) but ≤2cm (20mm), with evidence of invasion.
        - **T1c**: Tumor >2cm (20mm) but ≤3cm (30mm).
        - **T2a**: Tumor >3cm (30mm) but <4cm (40mm) or tumor involving visceral pleura, main bronchus (not carina), or atelectasis to hilum.
        - **T2b**: Tumor >4cm (40mm) but ≤5cm (50mm) or tumor involving visceral pleura, main bronchus (not carina), or atelectasis to hilum.
        - **T3**: Tumor >5cm (50mm) but ≤7cm (70mm) or invading chest wall, pericardium, phrenic nerve, or separate tumor nodule(s) in the same lobe.
        - **T4**: Tumor >7cm (70mm) or invading mediastinum, diaphragm, heart, great vessels, recurrent laryngeal nerve, trachea, esophagus, spine; or tumor nodule(s) in a different ipsilateral lobe.

        - In cases of **uncertainty** regarding invasion or measurements, choose the **most conservative stage**.
        - When describing tumor dimensions in the report, always distinguish between **total size** and **solid component size**, and use the solid component for staging where applicable.

        **II. Lymph Node Metastasis (N) Staging**:
        - **N0**: No regional lymph node metastasis.
          - Use only when lymph nodes are clearly normal without any suspicion of metastasis.
        - **N1**: Ipsilateral peribronchial or hilar lymph node metastasis.
          - Classify as **N1** if there is mention of "prominent hilar lymph nodes" or "suspicion of metastasis in the hilar nodes."
        - **N2**: Ipsilateral mediastinal or subcarinal lymph node metastasis.
          - Classify as **N2** if there is mention of "enlarged mediastinal lymph nodes" or "cannot rule out metastasis in mediastinal lymph nodes."
        - **N3**: Contralateral mediastinal, contralateral hilar, scalene, or supraclavicular lymph node metastasis.

        **III. Distant Metastasis (M) Staging**
        - **M0**: No distant metastasis.
          - **Key point**: Use this classification only when there is no evidence of distant metastasis, including pleural dissemination, malignant pleural effusion, or any organ involvement.
        - **M1a**: Pleural dissemination (malignant pleural or pericardial effusion, pleural nodules), or separate tumor nodules in the contralateral lobe.
          - **Key point**:
            - Include cases with pleural effusion confirmed as malignant or pleural-based nodules.
            - Also include separate nodules identified in the contralateral lung.
            - If additional evidence suggests metastases in other organs, classify as **M1c** instead.
        - **M1b**: Single-organ distant metastasis (outside lung/pleura).
          - **Key point**:
            - Classify as M1b if a single organ (e.g., liver, bone, brain, or adrenal gland) shows definitive evidence of metastasis.
            - Ensure that no metastases are present in other organs.
            - If multiple organs are involved, reclassify as **M1c**.
        - **M1c**: Multiple-organ distant metastases.
          - **Key point**:
            - Use this classification when metastases are identified in two or more organs (e.g., bone, liver, adrenal glands).
            - Include cases with pleural dissemination if other organs are also involved.
            - Examples include bilateral adrenal gland metastases or widespread bone lesions.

        If uncertain, select the stage that best fits the situation.\n\n

        **Output Requirement**:
        - Provide the TNM code in JSON format: {{'TNM': {{'T': '', 'N': '', 'M': ''}}}}.
        - Do not include reasoning, explanations, or any additional text outside the JSON format.

        **[Few-shot Examples]**\n
          {additional_cases}\n\n
        **[Testing Data]**\n
          The following are samples to be analyzed. Please provide the TNM staging results using the same logic."""
    },
    {
        "role": "user",
        "content": f"Input: {case}\n\nProvide the TNM staging result using the above logic."
    }
],
    temperature=0.2,
    max_tokens=50,
    top_p=0.8,
    frequency_penalty=0,
    presence_penalty=0,
    n=1
    )
    response = completion.choices[0].message.content.strip()
    result = parse_tnm_from_text(response)
    #json.loads(completion.choices[0].message.content.strip())

    return response, result

In [7]:
# response, result = classify_tnm(test_data['text'][22], n_shot=7, case_id = test_data['id'][22])
# print(response)
# print(result)

In [ ]:
def extract_tnm_fsv(text, case_id, n_votes, n_shot, max_retries=5):
    t_votes = []
    n_votes_list = []
    m_votes = []
    responses = []

    for _ in range(n_votes):
        retries = 0
        while retries < max_retries:
            try:
                # Call classify_tnm and collect the response and TNM values
                resp, tnm_result = classify_tnm(text, n_shot=n_shot, case_id=case_id)
                responses.append(resp)
                t_votes.append(tnm_result['TNM']['T'])
                n_votes_list.append(tnm_result['TNM']['N'])
                m_votes.append(tnm_result['TNM']['M'])
                break  # Successful extraction, exit retry loop
            except Exception as e:
                retries += 1
                print(f"Error encountered: {e}. Retry attempt {retries}/{max_retries}...")
        else:
            print(f"Failed to classify TNM for one vote after {max_retries} attempts.")

    # Perform hard voting for T, N, and M
    final_t = Counter(t_votes).most_common(1)[0][0] if t_votes else None
    final_n = Counter(n_votes_list).most_common(1)[0][0] if n_votes_list else None
    final_m = Counter(m_votes).most_common(1)[0][0] if m_votes else None

    # Return the concatenated responses and the final TNM result
    combined_response = "\n".join(responses) if responses else None
    return combined_response, final_t, final_n, final_m

# Initialize tqdm for pandas
tqdm.pandas()


n_shot = 7
n_votes = 3
test_data[['response', 'gen_T', 'gen_N', 'gen_M']] = test_data.progress_apply(
    lambda row: pd.Series(extract_tnm_fsv(row['text'], case_id=row['id'], n_shot=n_shot, n_votes=n_votes)),
    axis=1
)

In [ ]:
def calculate_and_print_accuracy(data, label):
    T_acc = (data['t'] == data['gen_T']).mean()
    N_acc = (data['n'] == data['gen_N']).mean()
    M_acc = (data['m'] == data['gen_M']).mean()
    joint_acc = ((data['t'] == data['gen_T']) & (data['n'] == data['gen_N']) & (data['m'] == data['gen_M'])).mean()

    print(f"{label} performance")
    print(f"=== Joint_acc: {joint_acc:.2%}")
    print(f"> T_acc: {T_acc:.2%}")
    print(f"> N_acc: {N_acc:.2%}")
    print(f"> M_acc: {M_acc:.2%}")

if n_votes == 1:
    w = "The result of {}shot without voting：".format(n_shot)
else:
    w = "The result of {}shot with {} voting：".format(n_shot,n_votes)
print(w)

# 計算全體數據的準確率
calculate_and_print_accuracy(test_data, "all")
print("")
# 計算驗證數據的準確率
all_val = test_data[test_data['status'] == 'val']
calculate_and_print_accuracy(all_val, "validation")
print("")
# 計算測試數據的準確率
all_test = test_data[test_data['status'] == 'test']
calculate_and_print_accuracy(all_test, "test")

In [ ]:
from datetime import datetime

def save_filtered_df(df, keep_text=False):
    # Define the columns to retain
    columns_to_keep = ['id', 'gen_T', 'gen_N', 'gen_M']

    # Add 'text' if keep_text is True
    if keep_text:
        columns_to_keep.append('text')

    # Filter and rename columns
    filtered_df = df[columns_to_keep].rename(columns={
        'gen_T': 't',
        'gen_N': 'n',
        'gen_M': 'm'
    })

    return filtered_df

filtered_df = save_filtered_df(test_data, keep_text=False)
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"../result/submit_test/resultGPT4o_en_{n_shot}s{n_votes}v{current_time}.csv"
filtered_df.to_csv(filename, sep=',', index=False)
print(f"File saved as: {filename}")

val Score (Joint acc / T_acc / N_acc / M_acc) <br>
90.74/94.44/100.0/96.30